<div style="
    text-align: center; 
    background: linear-gradient(135deg, #0062ff 0%, #00d4ff 100%); 
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; 
    color: white; 
    padding: 35px 20px; 
    border-radius: 15px; 
    box-shadow: 0 10px 25px rgba(0, 98, 255, 0.3);
    margin-bottom: 25px;">
    <div style="font-size: 35px; font-weight: 800; letter-spacing: 1.5px; text-transform: uppercase; line-height: 1.2;">
        Trực Quan Hóa Dữ Liệu - Đồ án cuối kỳ
    </div>
    <div style="font-size: 16px; font-weight: 500; margin-top: 10px; font-style: italic; opacity: 0.9;">
        "Phân tích xu hướng tuyển dụng và cơ hội nghề nghiệp ngành Công nghệ thông tin tại Việt Nam"
    </div>
    <div style="font-size: 18px; font-weight: 600; margin-top: 15px; border-top: 1px solid rgba(255,255,255,0.4); display: inline-block; padding-top: 10px; letter-spacing: 1px;">
        NHÓM 05 - FIT-HCMUS
    </div>
</div>

<div style="text-align: center; font-size: 40px; font-weight: bold;">
  TIỀN XỬ LÝ BỘ DỮ LIỆU
</div>

## **1. Đọc dữ liệu và tổng quan**

### **1.1. Mục tiêu**
Đọc bộ dữ liệu thô `vietnam_it_jobs_raw.csv` (11,439 tin × 18 cột) và khảo sát tổng quan: kiểu dữ liệu, tỷ lệ thiếu, mẫu dữ liệu — làm cơ sở cho các bước làm sạch tiếp theo.

### **1.2. Quy trình**
1. Đọc CSV từ `../data/`.
2. In `info()` và báo cáo tỷ lệ null từng cột.
3. Xem 3 dòng mẫu.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from bs4 import BeautifulSoup

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA_DIR = Path("../data")
RAW_PATH = DATA_DIR / "raw" / "vietnam_it_jobs_raw.csv"

df_raw = pd.read_csv(RAW_PATH)
print(f"Đã đọc: {RAW_PATH.name}")
print(f"Kích thước: {df_raw.shape[0]:,} dòng × {df_raw.shape[1]} cột\n")

print("=== THÔNG TIN CỘT ===")
df_raw.info()

print("\n=== TỶ LỆ THIẾU DỮ LIỆU ===")
null_report = df_raw.isna().sum().sort_values(ascending=False)
for col, n in null_report.items():
    pct = n * 100 // len(df_raw)
    bar = '#' * (pct // 5)
    print(f"  {col:22} {n:6,} ({pct:>2}%) {bar}")

df_raw.head(3)

Đã đọc: vietnam_it_jobs_raw.csv
Kích thước: 11,863 dòng × 18 cột

=== THÔNG TIN CỘT ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11863 entries, 0 to 11862
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   job_title            11863 non-null  object 
 1   company_name         10122 non-null  object 
 2   location             11837 non-null  object 
 3   salary               11861 non-null  object 
 4   salary_min           6550 non-null   float64
 5   salary_max           6940 non-null   float64
 6   skills               11267 non-null  object 
 7   description          11782 non-null  object 
 8   benefits             11512 non-null  object 
 9   employment_type      11815 non-null  object 
 10  industry             11782 non-null  object 
 11  posting_date         10122 non-null  object 
 12  snapshot_date        9339 non-null   object 
 13  deadline             10041 non-null  object 
 14

,job_title,company_name,location,salary,salary_min,salary_max,skills,description,benefits,employment_type,industry,posting_date,snapshot_date,deadline,source,url,crawled_at,experience_required
0,Automation Tester,Smilegate Vietnam,Quận Bình Thạnh,You'll love it,NaN,NaN,"Automation Test, Playwright, Scrum, Selenium, ...",Top 3 Reasons To Join Us\r\nBudget for individ...,Health and Wellness Premium health insurance S...,FULL_TIME,Information Technology,2026-05-27,2026-05-28,2026-07-01,ITviec (Wayback),https://itviec.com/it-jobs/automation-tester-s...,2026-06-08T01:38:59.289471,NaN
1,"Backend Developer Java, C++, OOP, Spring Boot",CÔNG TY CỔ PHẦN METAFORCE,Quận Nam Từ Liêm,Very Attractive!!!,NaN,NaN,"Java, Spring Boot, GitHub, OOP, C++, SQL",Top 3 Reasons To Join Us\r\nMức lương cạnh tra...,Chế độ lương thưởng Mức lương: thoả thuận Thử ...,FULL_TIME,Information Technology,2026-05-27,2026-05-28,2026-07-01,ITviec (Wayback),https://itviec.com/it-jobs/backend-developer-j...,2026-06-08T01:39:22.898350,NaN
2,Chuyen Vien QC Phan Mem QC Executive,Kingfoodmart,Quận 7,24–35 triệu,23.75,35.0,"QA QC, Automation Test, Tester","Top 3 Reasons To Join Us\r\nĐãi ngộ hấp dẫn, t...","Môi trường trẻ chuyên nghiệp, năng động, sáng ...",FULL_TIME,Information Technology,2026-05-27,2026-05-28,2026-07-01,ITviec (Wayback),https://itviec.com/it-jobs/chuyen-vien-qc-phan...,2026-06-08T01:41:34.142513,NaN


## **2. Làm sạch cơ bản**

### **2.1. Mục tiêu**
Loại bỏ dòng rác và chuẩn hóa định dạng văn bản trước khi xử lý sâu.

### **2.2. Quy trình**
1. Bỏ dòng không có `job_title`.
2. Chuẩn hóa khoảng trắng (gộp nhiều space, trim) cho các cột text.
3. Thay chuỗi rỗng bằng `NaN`.

In [2]:
df = df_raw.copy()
n0 = len(df)

# (1) Bỏ dòng không có tiêu đề công việc
df = df[df['job_title'].notna() & (df['job_title'].astype(str).str.strip() != '')]

# (2) Chuẩn hóa khoảng trắng cho các cột text
def norm_ws(s):
    if pd.isna(s):
        return np.nan
    return re.sub(r'\s+', ' ', str(s)).strip()

for col in ['job_title', 'company_name', 'location', 'salary', 'employment_type', 'description']:
    df[col] = df[col].apply(norm_ws)

# (3) Thay chuỗi rỗng bằng NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

print(f"Bỏ {n0 - len(df):,} dòng rác (thiếu job_title)")
print(f"Còn lại: {len(df):,} dòng")

Bỏ 0 dòng rác (thiếu job_title)
Còn lại: 11,863 dòng


## **3. Khử trùng lặp**

### **3.1. Mục tiêu**
Cùng một tin tuyển dụng có thể xuất hiện ở nhiều nguồn. Khử trùng để mỗi job chỉ còn một bản ghi.

### **3.2. Chiến lược**
* **Ưu tiên nguồn:** ITviec → TopDev → Vieclam24h → TopCV → VietJobs → JobsGO (nguồn tin cậy hơn được giữ khi trùng).
* **Job có công ty:** khử trùng theo `title + company + location` — gộp cùng một tin được lưu qua **nhiều URL khác nhau** (cross-source, cùng job đăng nhiều nơi).
* **Job không công ty (VietJobs/HTML):** khử theo URL nếu có, còn lại giữ nguyên để tránh gộp nhầm các tin khác nhau cùng tiêu đề.

In [3]:
n0 = len(df)

# Nguồn ưu tiên: số nhỏ = tin cậy hơn, được giữ khi trùng
SOURCE_PRIORITY = {'ITviec':1, 'TopDev':2, 'Vieclam24h':3, 'TopCV':4, 'VietJobs':5, 'JobsGO':6}
df['_src'] = df['source'].astype(str).str.split(' ').str[0]
df['_prio'] = df['_src'].map(SOURCE_PRIORITY).fillna(9)
df = df.sort_values('_prio', kind='stable')

def norm_key(s):
    if pd.isna(s):
        return ''
    return re.sub(r'\s+', ' ', str(s).lower().strip())

# Khóa nội dung: title + company + location + tháng đăng
df['_month'] = df['posting_date'].astype(str).str[:7]
df['_key'] = (df['job_title'].apply(norm_key) + '|'
              + df['company_name'].apply(norm_key) + '|'
              + df['location'].apply(norm_key) + '|'
              + df['_month'])

comp = df['company_name'].notna()

# (1) Job CÓ công ty: khử trùng theo title+company+location
#     -> gộp cùng một job được archive qua NHIỀU URL khác nhau (cross-source)
df_comp = df[comp].drop_duplicates(subset='_key', keep='first')

# (2) Job KHÔNG có công ty (VietJobs/HTML): theo URL nếu có, còn lại giữ nguyên
#     (tránh gộp nhầm các tin khác nhau cùng title nhưng thiếu company)
df_nc = df[~comp]
df_nc_url = df_nc[df_nc['url'].notna()].drop_duplicates(subset='url', keep='first')
df_nc_no  = df_nc[df_nc['url'].isna()]

df = (pd.concat([df_comp, df_nc_url, df_nc_no], ignore_index=True)
        .drop(columns=['_prio', '_key']))

print(f"Trước dedup : {n0:,}")
print(f"  Job có công ty (theo title+cty+loc) : giữ {len(df_comp):,}")
print(f"  Job không công ty                   : giữ {len(df_nc_url) + len(df_nc_no):,}")
print(f"Sau dedup  : {len(df):,}  (bỏ {n0 - len(df):,})")

Trước dedup : 11,863
  Job có công ty (theo title+cty+loc) : giữ 7,638
  Job không công ty                   : giữ 1,741
Sau dedup  : 9,379  (bỏ 2,484)


## **4. Chuẩn hóa địa điểm → tỉnh/thành**

### **4.1. Mục tiêu**
Cột `location` thô lẫn lộn cấp quận/huyện ("Quận 1", "Cầu Giấy") và hoa-thường khác nhau. Quy về **tỉnh/thành** chuẩn để vẽ bản đồ phân bố, kèm cột `vung_mien` (Bắc/Trung/Nam).

### **4.2. Quy tắc map**
* `Quận + số` → TP.HCM (Hà Nội không đánh số quận).
* Khớp tên quận/huyện đặc trưng của HN và HCM.
* Các tỉnh khác: khớp tên trực tiếp.
* Không khớp → `Khác`.

In [4]:
REMOTE_KEYS = ['remote', 'từ xa', 'tu xa', 'work from home', 'wfh', 'at home']
HCM_KEYS = ['hồ chí minh','ho chi minh','tphcm','tp hcm','hcm','sài gòn','sai gon',
            'thủ đức','thu duc','tân bình','bình thạnh','phú nhuận','gò vấp','tân phú',
            'bình tân','hóc môn','củ chi','nhà bè','bình chánh','cần giờ']
HN_KEYS  = ['hà nội','ha noi','hanoi','cầu giấy','đống đa','ba đình','hoàn kiếm',
            'hai bà trưng','thanh xuân','từ liêm','hà đông','long biên','tây hồ',
            'hoàng mai','gia lâm','đông anh','sóc sơn','thanh trì','hoài đức',
            'nam từ liêm','bắc từ liêm']

OTHER_PROV = {
    'đà nẵng':'Đà Nẵng','da nang':'Đà Nẵng',
    'hải phòng':'Hải Phòng','hai phong':'Hải Phòng',
    'cần thơ':'Cần Thơ','can tho':'Cần Thơ',
    'bình dương':'Bình Dương','binh duong':'Bình Dương',
    'đồng nai':'Đồng Nai','dong nai':'Đồng Nai',
    'bắc ninh':'Bắc Ninh','bac ninh':'Bắc Ninh',
    'hưng yên':'Hưng Yên','hung yen':'Hưng Yên',
    'quảng nam':'Quảng Nam','quang nam':'Quảng Nam',
    'huế':'Thừa Thiên Huế','thừa thiên':'Thừa Thiên Huế','hue':'Thừa Thiên Huế',
    'nha trang':'Khánh Hòa','khánh hòa':'Khánh Hòa','khanh hoa':'Khánh Hòa',
    'vũng tàu':'Bà Rịa - Vũng Tàu','bà rịa':'Bà Rịa - Vũng Tàu','vung tau':'Bà Rịa - Vũng Tàu',
    'quảng ninh':'Quảng Ninh','quang ninh':'Quảng Ninh',
    'thái nguyên':'Thái Nguyên','thai nguyen':'Thái Nguyên',
    'nghệ an':'Nghệ An','nghe an':'Nghệ An',
    'vĩnh phúc':'Vĩnh Phúc','vinh phuc':'Vĩnh Phúc',
    'hà nam':'Hà Nam','long an':'Long An','tiền giang':'Tiền Giang',
    'bình định':'Bình Định','đắk lắk':'Đắk Lắk','lâm đồng':'Lâm Đồng',
}

PROV_REGION = {
    'Từ xa / Remote':'Từ xa / Remote',
    'Hà Nội':'Bắc','Hải Phòng':'Bắc','Bắc Ninh':'Bắc','Hưng Yên':'Bắc','Quảng Ninh':'Bắc',
    'Thái Nguyên':'Bắc','Vĩnh Phúc':'Bắc','Hà Nam':'Bắc',
    'Đà Nẵng':'Trung','Thừa Thiên Huế':'Trung','Quảng Nam':'Trung','Khánh Hòa':'Trung',
    'Nghệ An':'Trung','Bình Định':'Trung','Đắk Lắk':'Trung','Lâm Đồng':'Trung',
    'TP.HCM':'Nam','Bình Dương':'Nam','Đồng Nai':'Nam','Cần Thơ':'Nam',
    'Bà Rịa - Vũng Tàu':'Nam','Long An':'Nam','Tiền Giang':'Nam',
}

_quan_so = re.compile(r'quận\s*\d+', re.I)

def map_province(loc):
    if pd.isna(loc):
        return 'Khác'
    s = str(loc).lower()
    for k in REMOTE_KEYS:
        if k in s:
            return 'Từ xa / Remote'
    if _quan_so.search(s):
        return 'TP.HCM'
    for k in HCM_KEYS:
        if k in s:
            return 'TP.HCM'
    for k in HN_KEYS:
        if k in s:
            return 'Hà Nội'
    for k, v in OTHER_PROV.items():
        if k in s:
            return v
    return 'Khác'
df['tinh_thanh'] = df['location'].apply(map_province)
df['vung_mien'] = df['tinh_thanh'].map(PROV_REGION).fillna('Khác')

print("=== PHÂN BỐ TỈNH/THÀNH ===")
print(df['tinh_thanh'].value_counts())
print("\n=== PHÂN BỐ VÙNG MIỀN ===")
print(df['vung_mien'].value_counts())

=== PHÂN BỐ TỈNH/THÀNH ===
tinh_thanh
TP.HCM               4596
Hà Nội               3416
Khác                  949
Đà Nẵng                80
Hải Phòng              46
Bắc Ninh               42
Đồng Nai               31
Thái Nguyên            29
Hưng Yên               28
Khánh Hòa              27
Lâm Đồng               20
Thừa Thiên Huế         19
Cần Thơ                18
Bà Rịa - Vũng Tàu      17
Quảng Ninh             15
Nghệ An                12
Đắk Lắk                12
Từ xa / Remote         12
Bình Dương              9
Long An                 1
Name: count, dtype: int64

=== PHÂN BỐ VÙNG MIỀN ===
vung_mien
Nam               4672
Bắc               3576
Khác               949
Trung              170
Từ xa / Remote      12
Name: count, dtype: int64


## **5. Chuẩn hóa mức lương**

### **5.1. Mục tiêu**
Quy lương về đơn vị **triệu VND**, loại bỏ giá trị rác và sinh thêm `luong_tb`, `loai_luong`.

### **5.2. Quy trình**
* Loại text rác không có số: `"You'll love it"`, `"Negotiable"`, `"Competitive"` → coi như thỏa thuận.
* Giá trị `>= 1000` xem là VND thô → chia 1,000,000.
* Lọc ngoại lai: giữ khoảng `1–500` triệu.
* `loai_luong`: `range` (có cả min-max) / `from` / `to` / `negotiable`.

In [5]:
def to_million(v):
    # Đưa giá trị về triệu VND; loại ngoại lai
    if v is None or pd.isna(v):
        return None
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    if v <= 0:
        return None
    if v >= 1000:            # VND thô -> triệu
        v = v / 1_000_000
    if v < 1 or v > 500:     # ngoại lai
        return None
    return round(v, 2)

def parse_salary_text(text):
    if not text or pd.isna(text):
        return None, None
    t = str(text).lower()
    if not re.search(r'\d', t):     # không có số -> rác/thỏa thuận
        return None, None
    is_usd = 'usd' in t or '$' in t
    is_annual = any(w in t for w in ['năm', 'year', 'annual', 'annum', 'yr'])
    t2 = t.replace('.', '').replace(',', '')
    nums = [float(n) for n in re.findall(r'\d+', t2) if float(n) > 0]
    if not nums:
        return None, None
    def convert_val(v):
        if is_annual:
            v = v / 12
        if is_usd:
            v = v * 25000 / 1000000
        else:
            if v >= 1000:
                v = v / 1_000_000
        if v < 1 or v > 500:
            return None
        return round(v, 2)
    if len(nums) >= 2:
        return convert_val(nums[0]), convert_val(nums[1])
    v = convert_val(nums[0])
    if re.search(r'từ|trên|from', t):
        return v, None
    if re.search(r'đến|tối đa|up to|under', t):
        return None, v
    return v, v
def resolve_salary(row):
    mn = to_million(row.get('salary_min'))
    mx = to_million(row.get('salary_max'))
    salary_text = str(row.get('salary') or '').lower()
    is_annual = any(w in salary_text for w in ['năm', 'year', 'annual', 'annum', 'yr'])
    if is_annual:
        if mn is not None: mn = round(mn / 12, 2)
        if mx is not None: mx = round(mx / 12, 2)
        if mn is not None and (mn < 1 or mn > 500): mn = None
        if mx is not None and (mx < 1 or mx > 500): mx = None
    if mn is None and mx is None:
        mn, mx = parse_salary_text(row.get('salary'))
    return pd.Series([mn, mx])
df[['luong_min', 'luong_max']] = df.apply(resolve_salary, axis=1)

def salary_avg(row):
    a, b = row['luong_min'], row['luong_max']
    has_a = a is not None and not pd.isna(a)
    has_b = b is not None and not pd.isna(b)
    if has_a and has_b:
        return round((a + b) / 2, 2)
    return a if has_a else (b if has_b else None)

def salary_type(row):
    a, b = row['luong_min'], row['luong_max']
    has_a = a is not None and not pd.isna(a)
    has_b = b is not None and not pd.isna(b)
    if has_a and has_b: return 'range'
    if has_a:           return 'from'
    if has_b:           return 'to'
    return 'negotiable'

df['luong_tb'] = df.apply(salary_avg, axis=1)
df['loai_luong'] = df.apply(salary_type, axis=1)

n_num = df['luong_tb'].notna().sum()
print(f"Có lương dạng số: {n_num:,} ({n_num*100//len(df)}%)")
print("\n=== LOẠI LƯƠNG ===")
print(df['loai_luong'].value_counts())
print("\n=== THỐNG KÊ LƯƠNG TRUNG BÌNH (triệu VND) ===")
print(df['luong_tb'].describe())

Có lương dạng số: 4,857 (51%)

=== LOẠI LƯƠNG ===
loai_luong
negotiable    4522
range         4472
to             363
from            22
Name: count, dtype: int64

=== THỐNG KÊ LƯƠNG TRUNG BÌNH (triệu VND) ===
count    4857.000000
mean       23.965403
std        18.729800
min         1.000000
25%        11.000000
50%        17.500000
75%        31.250000
max       250.000000
Name: luong_tb, dtype: float64


## **6. Chuẩn hóa kỹ năng**

### **6.1. Mục tiêu**
Cột `skills` thô lẫn 3 dạng: HTML (`<ul><li>...`), list-repr Python (`['Figma','HTML']`), và CSV thường. Làm sạch về chuỗi `"Python, ReactJS, AWS"` chuẩn.

### **6.2. Quy trình**
1. Bỏ thẻ HTML bằng BeautifulSoup.
2. Trích token trong dấu nháy nếu là list-repr.
3. Tách theo dấu phẩy/`;`/`|`, trim, bỏ token > 40 ký tự (câu văn).
4. Chuẩn hóa tên công nghệ theo từ điển canonical (reactjs→ReactJS…).

In [6]:
CANON = {
    'reactjs':'ReactJS','react':'ReactJS','react native':'React Native','nodejs':'NodeJS',
    'node js':'NodeJS','vuejs':'VueJS','vue':'VueJS','angular':'Angular','nextjs':'NextJS',
    'javascript':'JavaScript','typescript':'TypeScript','python':'Python','java':'Java',
    'golang':'Golang','c++':'C++','c#':'C#','.net':'.NET','dotnet':'.NET',
    'php':'PHP','laravel':'Laravel','ruby':'Ruby','rust':'Rust','kotlin':'Kotlin',
    'swift':'Swift','flutter':'Flutter','dart':'Dart','android':'Android','ios':'iOS',
    'sql':'SQL','mysql':'MySQL','postgresql':'PostgreSQL','postgres':'PostgreSQL',
    'mongodb':'MongoDB','redis':'Redis','oracle':'Oracle','elasticsearch':'Elasticsearch',
    'aws':'AWS','azure':'Azure','gcp':'GCP','google cloud':'GCP','docker':'Docker',
    'kubernetes':'Kubernetes','k8s':'Kubernetes','terraform':'Terraform','jenkins':'Jenkins',
    'ci/cd':'CI/CD','linux':'Linux','git':'Git','spring':'Spring','spring boot':'Spring Boot',
    'django':'Django','flask':'Flask','fastapi':'FastAPI','express':'ExpressJS',
    'html':'HTML','css':'CSS','tailwind':'TailwindCSS','bootstrap':'Bootstrap',
    'selenium':'Selenium','playwright':'Playwright','appium':'Appium','jira':'Jira',
    'machine learning':'Machine Learning','deep learning':'Deep Learning','nlp':'NLP',
    'tensorflow':'TensorFlow','pytorch':'PyTorch','pandas':'Pandas','spark':'Spark',
    'hadoop':'Hadoop','kafka':'Kafka','power bi':'Power BI','tableau':'Tableau',
    'excel':'Excel','sap':'SAP','erp':'ERP','figma':'Figma','devops':'DevOps',
    'data analysis':'Data Analysis','etl':'ETL','graphql':'GraphQL','rest api':'REST API',
    'microservices':'Microservices','agile':'Agile','scrum':'Scrum',
}
_CANON_ITEMS = sorted(CANON.items(), key=lambda kv: -len(kv[0]))

def clean_skill_string(raw):
    if pd.isna(raw):
        return np.nan
    s = str(raw)
    if '<' in s:                                  # bỏ HTML
        s = BeautifulSoup(s, 'html.parser').get_text(' ', strip=True)
    tokens = []
    if '[' in s and ("'" in s or '"' in s):       # list-repr
        tokens = re.findall(r"['\"]([^'\"]+)['\"]", s)
    if not tokens:
        tokens = re.split(r'[,;|/]', s)
    cleaned, seen = [], set()
    for t in tokens:
        t = re.sub(r'\s+', ' ', t).strip(' .-')
        low = t.lower()
        if not t or len(t) > 40:                  # câu văn -> trích tech keyword
            for k, v in _CANON_ITEMS:
                if re.search(rf'\b{re.escape(k)}\b', low) and v.lower() not in seen:
                    cleaned.append(v); seen.add(v.lower())
            continue
        canon = CANON.get(low, t)
        if canon.lower() not in seen:
            cleaned.append(canon); seen.add(canon.lower())
    return ', '.join(cleaned) if cleaned else np.nan

df['ky_nang'] = df['skills'].apply(clean_skill_string)

n_sk = df['ky_nang'].notna().sum()
print(f"Có kỹ năng: {n_sk:,} ({n_sk*100//len(df)}%)")
print("\n=== VÍ DỤ ===")
for s in df['ky_nang'].dropna().head(8):
    print('  -', s[:80])

Có kỹ năng: 8,785 (93%)

=== VÍ DỤ ===
  - Automation Test, Playwright, Scrum, Selenium, Appium, Agile
  - Java, Spring Boot, GitHub, OOP, C++, SQL
  - QA QC, Automation Test, Tester
  - C++, Windows, Unity, C#, Fresher Accepted
  - Machine Learning, Cloud, SQL, Python, English, AI
  - Generative AI, LLM, AI, Solution Architecture, Software Architecture, English
  - Data Engineer, Database, SQL, Unix, Linux, Business Analysis
  - ReactJS, CSS 3, API, HTML5, JavaScript, React Native


## **7. Phân loại vị trí & cấp độ kinh nghiệm**

### **7.1. Mục tiêu**
Sinh 2 cột phân tích cốt lõi và **lọc tin không thuộc IT**:
* `nhom_vi_tri`: 10 nhóm chuẩn IT + nhãn `Không phải IT` cho tin lọt từ nguồn (sales, CSKH, kế toán, cơ khí…).
* `cap_do_kinh_nghiem`: Intern / Fresher / Junior / Middle / Senior.

### **7.2. Quy tắc**
* **Chuẩn hóa không dấu:** title có cả dạng có dấu ("Vận hành") và không dấu ("Van hanh") → bỏ dấu trước khi match để không sót.
* **Lọc phi-IT:** nếu title khớp từ khóa nghề phổ thông (kinh doanh, CSKH, kế toán, cơ khí, nấu bia…) **và** không có tín hiệu IT rõ → gán `Không phải IT` (sẽ loại ở bước 9).
* **Nhóm vị trí:** regex trên `title + ky_nang`, thứ tự **cụ thể → tổng quát** (Cybersecurity, QA, AI/ML… trước; Software Development gần cuối).
* **Cấp độ:** thứ tự (1) từ khóa cấp bậc trong title → (2) **số năm trong title** → (3) `experience_required` → (4) số năm trong mô tả → (5) từ khóa cấp bậc trong mô tả. Hàm `extract_years` bắt nhiều dạng (mốc tối thiểu "từ/tối thiểu/ít nhất X", cộng thêm "X+/trở lên", khoảng "X–Y/X đến Y", "X yoe"), lấy **mốc tối thiểu**; "không yêu cầu/entry-level/được đào tạo"→0. Ánh xạ 0→Fresher, 1→Junior, 2-4→Middle, ≥5→Senior. Còn ~56% "Không rõ" (bản chất tin không nêu kinh nghiệm).

In [7]:
import unicodedata

def no_accent(s):
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.replace('đ', 'd').replace('Đ', 'd').lower()

# Tín hiệu IT rõ ràng -> bảo vệ khỏi bị lọc nhầm sang "Không phải IT"
IT_SIGNAL = re.compile(
    r'phan mem|software|developer|engineer|lap trinh|devops|\bdata\b|may tinh|'
    r'he thong thong tin|\bserver\b|helpdesk|sysadmin|quan tri mang|quan tri he thong|'
    r'network|\bweb\b|\bapp\b|\bui\b|\bux\b|ui/ux|ui-ux|mo hinh|\bit\b|cntt|'
    r'reactjs|javascript|typescript|front-end|back-end|fullstack')

# Nghề phổ thông / không thuộc IT
NON_IT = re.compile(
    r'kinh doanh|\bsale|ban hang|telesale|business development|cham soc khach hang|cskh|'
    r'giao dich vien|tong dai|tu van vien|tu van khach hang|tu van tai chinh|\btin dung\b|'
    r'ke toan|kiem toan|hanh chinh|nhan su|tuyen dung|thu ky|van thu|phap ly|phap che|'
    r'marketing|content|truyen thong|copywriter|bien tap|do hoa|graphic|'
    r'nau bia|brewing|dau bep|nau an|pha che|barista|phuc vu|co khi|dien lanh|dien nhe|'
    r'\boto\b|ky su xay dung|xay dung cong trinh|xay dung dan dung|noi that|thi cong|'
    r'lap dat camera|lap mang|thay the linh kien|bao tri may|bao duong may|'
    r'lai xe|bao ve|tap vu|giao hang|shipper|thu kho|y te|dieu duong|\bduoc\b|'
    r'giao vien|gia su|phien dich|bien dich')

# Quy tắc nhóm vị trí (không dấu), thứ tự cụ thể -> tổng quát
CATEGORY_RULES = [
    ('Cybersecurity',
     r'security|cybersecurity|bao mat|an ninh mang|\bsoc\b|siem|devsecops|pentest|infosec'),
    ('QA / Testing',
     r'\bqa\b|\bqc\b|tester|testing|kiem thu|automation test|manual test|selenium|sdet'),
    ('AI / ML / Data Science',
     r'\bai\b|machine learning|\bml\b|deep learning|data scientist|data analyst|data analysis|'
     r'computer vision|\bnlp\b|generative|\bllm\b|khoa hoc du lieu|phan tich du lieu|\bbi\b'),
    ('Data Engineering / Database',
     r'data engineer|database|\bdba\b|big data|data warehouse|\betl\b|data platform|'
     r'ky su du lieu|co so du lieu|data governance'),
    ('Cloud / DevOps / SRE',
     r'devops|\bsre\b|cloud|kubernetes|infrastructure|platform engineer|site reliability'),
    ('Mobile / Game / Embedded',
     r'mobile|android|\bios\b|flutter|react native|game|embedded|firmware|nhung|'
     r'3d modeler|3d artist|unity|unreal'),
    ('Product / Business / UX',
     r'product owner|product manager|business analyst|\bba\b|\bux\b|\bui\b|product designer|'
     r'phan tich nghiep vu|head of product'),
    ('Management / Architecture',
     r'project manager|scrum master|tech lead|team lead|\bcto\b|\bcio\b|architect|kien truc su|'
     r'quan ly du an|truong nhom|\bbrse\b|\bmanager\b|tu van giai phap|solution consultant|it consultant'),
    ('IT Support / ERP',
     r'it support|helpdesk|sysadmin|system admin|quan tri he thong|quan tri mang|\berp\b|\bsap\b|'
     r'sfdc|it operations|ho tro ky thuat|nhan vien it|ky thuat it|ky thuat vien may tinh|'
     r'chuyen vien it|network admin|vien thong|application support|van hanh ung dung|'
     r'van hanh he thong|van hanh phan mem|appops|\bit\b'),
    ('Software Development',
     r'developer|\bdev\b|engineer|programmer|\bdevelopment\b|backend|back-end|frontend|front-end|'
     r'fullstack|full-stack|software|lap trinh|\bphp\b|\bjava\b|\.net|nodejs|python|reactjs|'
     r'javascript|typescript|angular|html5|c#|web developer|ky su phan mem|mainframe|cobol|'
     r'middleware|phat trien phan mem|phat trien ung dung|phat trien he thong'),
]
CATEGORY_RULES = [(name, re.compile(pat)) for name, pat in CATEGORY_RULES]

def classify_category(row):
    title = no_accent(row.get('job_title', ''))
    # Lọc phi-IT: có dấu hiệu nghề phổ thông và KHÔNG có tín hiệu IT
    if not IT_SIGNAL.search(title) and NON_IT.search(title):
        return 'Không phải IT'
    text = title + ' ' + no_accent(row.get('ky_nang', '') or '')
    for name, rgx in CATEGORY_RULES:
        if rgx.search(text):
            return name
    return 'Other'

df['nhom_vi_tri'] = df.apply(classify_category, axis=1)

def parse_exp_years(s):
    if pd.isna(s):
        return None
    low = no_accent(s)
    m = re.search(r'(\d+)\s*(nam|year)', low)
    if m:
        return int(m.group(1))
    if re.search(r'khong yeu cau|khong can|no exp|fresher|intern', low):
        return 0
    return None

def years_to_level(y):
    if y == 0: return 'Fresher'
    if y < 2:  return 'Junior'
    if y < 5:  return 'Middle'
    return 'Senior'

# Bắt SỐ NĂM kinh nghiệm theo nhiều dạng: "3 yoe", ">3yoe", "from 3 years",
# "2+ years", "minimum 8-10 years", "5 năm trở lên", "1 đến 3 năm",
# "X năm kinh nghiệm", "kinh nghiệm X năm"...  (lấy mốc tối thiểu)
_YEAR_PATS = [re.compile(p) for p in [
    r'>?\s*(\d+)\s*\+?\s*yoe',
    r'(?:tu|from|toi thieu|it nhat|minimum|min|at least|tren|over|>)\s*(\d+)\s*\+?\s*(?:nam|years?)\b',
    r'(\d+)\s*\+\s*(?:nam|years?)\b',
    r'(\d+)\s*(?:nam|years?)\s*tro len',
    r'(\d+)\s*(?:[-~]|den)\s*\d+\s*(?:nam|years?|yoe)',
    r'(\d+)\s*(?:nam|years?)[^.]{0,18}(?:kinh nghiem|experience|exp\b)',
    r'(?:kinh nghiem|experience|\bexp)[^.]{0,18}(\d+)\s*(?:nam|years?)',
]]

def extract_years(text_na):
    # Trích số năm KN nhỏ nhất từ chuỗi ĐÃ bỏ dấu; None nếu không có
    found = []
    for p in _YEAR_PATS:
        for m in p.finditer(text_na):
            v = int(m.group(1))
            if 0 <= v <= 40:
                found.append(v)
    return min(found) if found else None

# Từ khóa cấp bậc (không phải số năm) — dùng cho mô tả
_NOEXP  = re.compile(r'khong yeu cau kinh nghiem|khong can kinh nghiem|no experience|entry.?level|duoc dao tao|fresher accepted|moi tot nghiep|moi ra truong|sinh vien nam cuoi')
_INTERN = re.compile(r'\bintern\b|internship|thuc tap')
_SENIOR = re.compile(r'\bsenior\b')
_MIDDLE = re.compile(r'middle\s*level|mid-?level|middle\s*(?:developer|engineer|dev)\b')
_JUNIOR = re.compile(r'\bjunior\b')

def level_from_keywords(d):
    if _INTERN.search(d): return 'Intern'
    if _NOEXP.search(d):  return 'Fresher'
    if _SENIOR.search(d): return 'Senior'
    if _MIDDLE.search(d): return 'Middle'
    if _JUNIOR.search(d): return 'Junior'
    return None

def classify_level(row):
    title = no_accent(row.get('job_title', ''))
    title_clean = re.sub(r'\bmoi\s+truong\b', '', title)
    # 1. Cấp bậc nêu thẳng trong title (chú ý \bmiddle\b để không dính "middleware")
    if re.search(r'intern|thuc tap|\btts\b', title_clean):                                       return 'Intern'
    if re.search(r'fresher|moi ra truong|no experience|entry.?level', title_clean):              return 'Fresher'
    if re.search(r'senior|\bsr\b|lead|principal|\btruong\s+(?:nhom|phong|bo\s+phan)\b|architect|manager|\bcto\b', title_clean): return 'Senior'
    if re.search(r'\bmiddle\b|\bmid\b|mid-level', title_clean):                                  return 'Middle'
    if re.search(r'junior|\bjr\b', title_clean):                                                 return 'Junior'
    # 2. Số năm trong title
    y = extract_years(title_clean)
    if y is not None:
        return years_to_level(y)
    # 3. Trường experience_required
    y = parse_exp_years(row.get('experience_required'))
    if y is not None:
        return years_to_level(y)
    # 4. Số năm trong mô tả → từ khóa cấp bậc trong mô tả
    desc = row.get('description')
    if pd.notna(desc):
        d = no_accent(desc)
        y = extract_years(d)
        if y is not None:
            return years_to_level(y)
        kw = level_from_keywords(d)
        if kw:
            return kw
    return 'Không rõ'

df['cap_do_kinh_nghiem'] = df.apply(classify_level, axis=1)

print("=== PHÂN NHÓM VỊ TRÍ ===")
print(df['nhom_vi_tri'].value_counts())
print("\n=== CẤP ĐỘ KINH NGHIỆM ===")
print(df['cap_do_kinh_nghiem'].value_counts())

=== PHÂN NHÓM VỊ TRÍ ===
nhom_vi_tri
Software Development           2503
Other                           982
AI / ML / Data Science          937
Không phải IT                   927
Mobile / Game / Embedded        753
IT Support / ERP                695
QA / Testing                    499
Cloud / DevOps / SRE            473
Management / Architecture       460
Product / Business / UX         457
Data Engineering / Database     378
Cybersecurity                   315
Name: count, dtype: int64

=== CẤP ĐỘ KINH NGHIỆM ===
cap_do_kinh_nghiem
Không rõ    5280
Senior      1544
Middle      1162
Junior       709
Fresher      442
Intern       242
Name: count, dtype: int64


## **8. Chuẩn hóa hình thức làm việc & thời gian**

### **8.1. Mục tiêu**
* Gộp `employment_type` (lẫn enum tiếng Anh `FULL_TIME` và tiếng Việt "Toàn thời gian") về 4 nhóm chuẩn.
* Làm sạch `posting_date`: bỏ ngày rác (`1970-01-01`, ngày tương lai), sinh `thang_dang` (YYYY-MM).

### **8.2. Quy trình**
1. Map hình thức → Full-time / Part-time / Internship / Contract / Khác.
2. Parse ngày, giữ khoảng hợp lệ `2022-01` → `2026-06`.

In [8]:
def map_employment(s):
    if pd.isna(s):
        return 'Không rõ'
    t = str(s).lower()
    if 'intern' in t or 'thực tập' in t:               return 'Internship'
    if 'part' in t or 'bán thời gian' in t:            return 'Part-time'
    if 'contract' in t or 'temporary' in t or 'hợp đồng' in t: return 'Contract'
    if 'full' in t or 'toàn thời gian' in t:           return 'Full-time'
    return 'Khác'

df['hinh_thuc_lam_viec'] = df['employment_type'].apply(map_employment)

# Làm sạch ngày đăng
dt = pd.to_datetime(df['posting_date'], errors='coerce')
valid = (dt >= '2022-01-01') & (dt <= '2026-06-30')
dt = dt.where(valid)
df['ngay_dang'] = dt.dt.strftime('%Y-%m-%d')
df['thang_dang'] = dt.dt.strftime('%Y-%m')

print("=== HÌNH THỨC LÀM VIỆC ===")
print(df['hinh_thuc_lam_viec'].value_counts())
print(f"\nNgày đăng hợp lệ: {df['ngay_dang'].notna().sum():,}")
print("\n=== PHÂN BỐ THEO THÁNG ===")
print(df['thang_dang'].value_counts().sort_index())

=== HÌNH THỨC LÀM VIỆC ===
hinh_thuc_lam_viec
Full-time     7113
Khác          2047
Internship     112
Không rõ        45
Contract        31
Part-time       31
Name: count, dtype: int64

Ngày đăng hợp lệ: 7,634

=== PHÂN BỐ THEO THÁNG ===
thang_dang
2022-06      1
2022-08      1
2022-10      3
2022-11      2
2022-12      1
2023-01     47
2023-02     62
2023-03    108
2023-04     38
2023-05    246
2023-06     85
2023-07    199
2023-08     13
2023-09     67
2023-10     12
2023-11     37
2023-12     11
2024-01     18
2024-02     10
2024-03     21
2024-04     35
2024-05     43
2024-06     24
2024-07     23
2024-08     50
2024-09     28
2024-10     77
2024-11    135
2024-12    224
2025-01    302
2025-02    270
2025-03    307
2025-04    215
2025-05    220
2025-06    455
2025-07    638
2025-08    381
2025-09    291
2025-10    526
2025-11    527
2025-12    428
2026-01    236
2026-02    104
2026-03     99
2026-04     93
2026-05    176
2026-06    745
Name: count, dtype: int64


## **9. Đánh giá & xuất file**

### **9.1. Mục tiêu**
Loại tin `Không phải IT`, chọn 18 cột cuối (tên tiếng Việt không dấu), đánh giá tỷ lệ thiếu sau xử lý và xuất `vietnam_it_jobs_processed.csv`. (Biểu đồ trực quan để dành cho `eda.ipynb`.)

### **9.2. Schema đầu ra (18 cột)**
`ten_cong_viec`, `ten_cong_ty`, `nhom_vi_tri`, `cap_do_kinh_nghiem`, `tinh_thanh`, `vung_mien`, `luong_goc`, `luong_min`, `luong_max`, `luong_tb`, `loai_luong`, `ky_nang`, `mo_ta`, `hinh_thuc_lam_viec`, `ngay_dang`, `thang_dang`, `nguon`, `url`.

In [9]:
# Loại bỏ tin không thuộc IT (đã gán nhãn ở bước 7)
n_before = len(df)
n_nonit = (df['nhom_vi_tri'] == 'Không phải IT').sum()
df = df[df['nhom_vi_tri'] != 'Không phải IT'].copy()
print(f"Loại {n_nonit:,} tin KHÔNG phải IT  →  còn {len(df):,} tin IT\n")

df_final = df.rename(columns={
    'job_title':   'ten_cong_viec',
    'company_name':'ten_cong_ty',
    'salary':      'luong_goc',
    'description': 'mo_ta',
})
df_final['nguon'] = df['_src']

FINAL_COLS = [
    'ten_cong_viec', 'ten_cong_ty', 'nhom_vi_tri', 'cap_do_kinh_nghiem',
    'tinh_thanh', 'vung_mien', 'luong_goc', 'luong_min', 'luong_max', 'luong_tb',
    'loai_luong', 'ky_nang', 'mo_ta', 'hinh_thuc_lam_viec', 'ngay_dang', 'thang_dang',
    'nguon', 'url',
]
df_final = df_final[FINAL_COLS]

print("=== TỶ LỆ THIẾU SAU XỬ LÝ ===")
for c in df_final.columns:
    n = df_final[c].isna().sum()
    pct = n * 100 // len(df_final)
    print(f"  {c:22} thiếu {n:5,} ({pct:>2}%)")

PROCESSED_DATA_DIR = DATA_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = PROCESSED_DATA_DIR / "vietnam_it_jobs_processed.csv"
df_final.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print(f"\nĐã xuất: {OUT_PATH}")
print(f"  {len(df_final):,} dòng × {len(df_final.columns)} cột")
print(f"  Cột: {list(df_final.columns)}")

Loại 927 tin KHÔNG phải IT  →  còn 8,452 tin IT

=== TỶ LỆ THIẾU SAU XỬ LÝ ===
  ten_cong_viec          thiếu     0 ( 0%)
  ten_cong_ty            thiếu 1,376 (16%)
  nhom_vi_tri            thiếu     0 ( 0%)
  cap_do_kinh_nghiem     thiếu     0 ( 0%)
  tinh_thanh             thiếu     0 ( 0%)
  vung_mien              thiếu     0 ( 0%)
  luong_goc              thiếu     2 ( 0%)
  luong_min              thiếu 4,635 (54%)
  luong_max              thiếu 4,315 (51%)
  luong_tb               thiếu 4,295 (50%)
  loai_luong             thiếu     0 ( 0%)
  ky_nang                thiếu   443 ( 5%)
  mo_ta                  thiếu    64 ( 0%)
  hinh_thuc_lam_viec     thiếu     0 ( 0%)
  ngay_dang              thiếu 1,380 (16%)
  thang_dang             thiếu 1,380 (16%)
  nguon                  thiếu     0 ( 0%)
  url                    thiếu 1,376 (16%)

Đã xuất: ..\data\processed\vietnam_it_jobs_processed.csv
  8,452 dòng × 18 cột
  Cột: ['ten_cong_viec', 'ten_cong_ty', 'nhom_vi_tri', 'cap_do_kinh